# 1.HumanInTheLoopMiddleware中间件

## 1.1举例的过程1：工具调用的中断

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

summary_model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
    profile = {"max_input_tokens":64000}   # 务必显式赋值
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from rich import print as rprint


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


agent = create_agent(
    model = "deepseek-flash",
    tools = [get_news, get_weather, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather":True,  # 调用前必须终端等待审批
                "get_news":True,   # 同上
                "read_email_tool":False,   # 不需要终端，直接执行
                "send_email_tool":{   # 高级配置
                    "allowed_decisions":["approve", "reject"],
                    "description":"发送邮件中断了！"
                }
            },
            description_prefix="中断了！"
        )
    ]
)

config = {"configurable":{"thread_id":"1"}}   # 恢复中断时要用同一个thread_id

# 第一次调用：会暂停在发送邮件前
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请帮我查询今天北京的天气"
                                 "查询今日新闻"
                                 "查看ID为 'sk2131421' 的邮件内容，"
                                 "内容是：'你好啊'"
                                 "向15641685664@qq.com发送邮件，标题是'哈哈'，内"
                                 "同时做这四件事")
        ]
    },
    config=config,
)

print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)
print("========= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()

# 关键：看中断信息
interrupts = response.get("__interrupt__", [])
print("========= interrupts =========")
rprint(interrupts)

## 举例的过程2：指明工具调用请求策略

In [ ]:
from langgraph.types import Command

weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args":{"city": "上海市", "is_forcast": True}
    }
}

news_decision = {
    "type":"approve"
}

send_email_decision = {
    "type":"approve"
}


decisions = {
    "decisions": []
}

interrupts = response.get("__interrupt__", [])
action_requests = interrupts[0].value["action_requests"]

for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts:
    resumed_res = agent.invoke(
        Command(resume=decisions),
        config = config
    )

    for msg in resumed_res["messages"]:
        msg.pretty_print()